# KKBox Churn Prediction — Data Processing

Extracts the raw Kaggle `.7z` archives in `data/raw/` and writes compact, typed parquet files to `data/processed/`.

Notes:
- `user_logs.csv` (~30GB uncompressed, ~400M rows) is streamed through a named pipe straight from the 7z decoder into chunked parquet writes, so the full CSV is never materialized on disk (it wouldn't fit alongside the parquet output on this machine).
- All other archives are small enough to extract to a scratch dir, convert, and delete.
- `sample_submission*.csv.7z` files are Kaggle submission templates, not modeling data, so they're skipped.

In [1]:
import os
import shutil
import threading
import time

import pandas as pd
import py7zr
import pyarrow as pa
import pyarrow.parquet as papq

RAW_DIR = os.path.join(os.getcwd(), "data", "raw")
PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
TMP_DIR = os.path.join(RAW_DIR, "_tmp_extract")

CHUNKSIZE = 5_000_000

os.makedirs(PROCESSED_DIR, exist_ok=True)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

In [2]:
def extract(archive_name):
    """Extract a small/medium archive to a scratch dir and return the csv path."""
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)
    archive_path = os.path.join(RAW_DIR, archive_name)
    log(f"Extracting {archive_name} ...")
    with py7zr.SevenZipFile(archive_path, "r") as z:
        names = z.getnames()
        z.extractall(path=TMP_DIR)
    # archive may nest the csv under a subfolder (e.g. data/churn_comp_refresh/*)
    extracted_path = os.path.join(TMP_DIR, names[0])
    log(f"Extracted to {extracted_path}")
    return extracted_path


def cleanup():
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)

## members_v3 / train / train_v2

Small enough to load fully into memory.

In [3]:
def convert_members():
    out_path = os.path.join(PROCESSED_DIR, "members.parquet")
    if os.path.exists(out_path):
        log(f"Skipping members_v3.csv.7z, {out_path} already exists")
        return
    csv_path = extract("members_v3.csv.7z")
    dtypes = {
        "msno": "string",
        "city": "int8",
        "bd": "int16",
        "gender": "category",
        "registered_via": "int8",
        "registration_init_time": "int32",
    }
    df = pd.read_csv(csv_path, dtype=dtypes)
    df.to_parquet(out_path, engine="pyarrow", compression="zstd", index=False)
    log(f"Wrote {out_path} ({len(df):,} rows)")
    cleanup()


def convert_train(archive_name, out_name):
    out_path = os.path.join(PROCESSED_DIR, out_name)
    if os.path.exists(out_path):
        log(f"Skipping {archive_name}, {out_path} already exists")
        return
    csv_path = extract(archive_name)
    dtypes = {"msno": "string", "is_churn": "int8"}
    df = pd.read_csv(csv_path, dtype=dtypes)
    df.to_parquet(out_path, engine="pyarrow", compression="zstd", index=False)
    log(f"Wrote {out_path} ({len(df):,} rows)")
    cleanup()

In [4]:
convert_members()
convert_train("train.csv.7z", "train.parquet")
convert_train("train_v2.csv.7z", "train_v2.parquet")

[17:42:47] Skipping members_v3.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/members.parquet already exists
[17:42:47] Skipping train.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/train.parquet already exists
[17:42:47] Skipping train_v2.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/train_v2.parquet already exists


## transactions / transactions_v2 / user_logs_v2

Medium-sized files: extract to a scratch dir, then stream through pandas in chunks and write incrementally with a `pyarrow.ParquetWriter` (keeps peak memory low).

In [5]:
TRANSACTIONS_DTYPES = {
    "msno": "string",
    "payment_method_id": "int8",
    "payment_plan_days": "int16",
    "plan_list_price": "int16",
    "actual_amount_paid": "int32",
    "is_auto_renew": "int8",
    "transaction_date": "int32",
    "membership_expire_date": "int32",
    "is_cancel": "int8",
}

USER_LOGS_DTYPES = {
    "msno": "string",
    "date": "int32",
    "num_25": "int32",
    "num_50": "int32",
    "num_75": "int32",
    "num_985": "int32",
    "num_100": "int32",
    "num_unq": "int32",
    "total_secs": "float32",
}

PA_TYPES = {
    "string": pa.string(),
    "int8": pa.int8(),
    "int16": pa.int16(),
    "int32": pa.int32(),
    "float32": pa.float32(),
}

In [6]:
def convert_chunked(archive_name, out_name, dtypes):
    out_path = os.path.join(PROCESSED_DIR, out_name)
    if os.path.exists(out_path):
        log(f"Skipping {archive_name}, {out_path} already exists")
        return
    csv_path = extract(archive_name)
    schema = pa.schema([(col, PA_TYPES[dt]) for col, dt in dtypes.items()])
    writer = None
    total_rows = 0
    try:
        reader = pd.read_csv(csv_path, dtype=dtypes, chunksize=CHUNKSIZE)
        for i, chunk in enumerate(reader):
            table = pa.Table.from_pandas(chunk, schema=schema, preserve_index=False)
            if writer is None:
                writer = papq.ParquetWriter(out_path, table.schema, compression="zstd")
            writer.write_table(table)
            total_rows += len(chunk)
            log(f"  {archive_name}: chunk {i} written, {total_rows:,} rows so far")
    finally:
        if writer is not None:
            writer.close()
    log(f"Wrote {out_path} ({total_rows:,} rows)")
    cleanup()

In [7]:
convert_chunked("transactions.csv.7z", "transactions.parquet", TRANSACTIONS_DTYPES)
convert_chunked("transactions_v2.csv.7z", "transactions_v2.parquet", TRANSACTIONS_DTYPES)
convert_chunked("user_logs_v2.csv.7z", "user_logs_v2.parquet", USER_LOGS_DTYPES)

[17:42:47] Skipping transactions.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/transactions.parquet already exists
[17:42:47] Skipping transactions_v2.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/transactions_v2.parquet already exists
[17:42:47] Skipping user_logs_v2.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/user_logs_v2.parquet already exists


## user_logs (~30GB uncompressed, ~400M rows)

Too large to extract to disk alongside its parquet output given available free space. Instead, the 7z archive's single member is decompressed by `py7zr` directly into a named pipe (FIFO) in a background thread, while pandas reads chunks from that pipe on the main thread — the raw CSV bytes are never fully materialized on disk.

`py7zr` does a final `seek(0)` on the output file to verify its CRC after writing, which isn't supported on a FIFO and always raises `OSError` at that point — by then all bytes have already been written and consumed, so the error is expected and safe to ignore.

In [8]:
def convert_streaming(archive_name, out_name, dtypes):
    out_path = os.path.join(PROCESSED_DIR, out_name)
    if os.path.exists(out_path):
        log(f"Skipping {archive_name}, {out_path} already exists")
        return
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)
    archive_path = os.path.join(RAW_DIR, archive_name)

    with py7zr.SevenZipFile(archive_path, "r") as z:
        names = z.getnames()
    assert len(names) == 1, names
    fifo_path = os.path.join(TMP_DIR, names[0])
    os.mkfifo(fifo_path)

    def extract_worker():
        try:
            with py7zr.SevenZipFile(archive_path, "r") as z:
                z.extractall(path=TMP_DIR)
        except OSError:
            pass  # expected: seek(0) on the fifo for post-extract CRC check

    thread = threading.Thread(target=extract_worker)
    thread.start()

    schema = pa.schema([(col, PA_TYPES[dt]) for col, dt in dtypes.items()])
    writer = None
    total_rows = 0
    try:
        reader = pd.read_csv(fifo_path, dtype=dtypes, chunksize=CHUNKSIZE)
        for i, chunk in enumerate(reader):
            table = pa.Table.from_pandas(chunk, schema=schema, preserve_index=False)
            if writer is None:
                writer = papq.ParquetWriter(out_path, table.schema, compression="zstd")
            writer.write_table(table)
            total_rows += len(chunk)
            log(f"  {archive_name}: chunk {i} written, {total_rows:,} rows so far")
    finally:
        if writer is not None:
            writer.close()
    thread.join()
    log(f"Wrote {out_path} ({total_rows:,} rows)")
    cleanup()

In [9]:
convert_streaming("user_logs.csv.7z", "user_logs.parquet", USER_LOGS_DTYPES)
log("All done.")

[17:42:48] Skipping user_logs.csv.7z, /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/user_logs.parquet already exists
[17:42:48] All done.


## Verify outputs

In [10]:
import pyarrow.parquet as papq

for fname in sorted(os.listdir(PROCESSED_DIR)):
    path = os.path.join(PROCESSED_DIR, fname)
    pf = papq.ParquetFile(path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"{fname:30s} rows={pf.metadata.num_rows:>12,} cols={pf.metadata.num_columns} size={size_mb:,.1f}MB")

members.parquet                rows=   6,769,473 cols=6 size=236.3MB
train.parquet                  rows=     992,931 cols=2 size=33.0MB
train_v2.parquet               rows=     970,960 cols=2 size=32.3MB
transactions.parquet           rows=  21,547,746 cols=9 size=783.8MB
transactions_v2.parquet        rows=   1,431,009 cols=9 size=52.2MB
user_logs.parquet              rows= 392,106,543 cols=9 size=6,419.7MB
user_logs_v2.parquet           rows=  18,396,362 cols=9 size=769.0MB
